# KNN-based DDoS Detection & Mitigation in SDN
## Empirical Evaluation Notebook (Kaggle Edition)

This notebook executes the complete statistical and empirical evaluation pipeline for the research framework:
- **Baseline KNN**: Standard K-Nearest Neighbors classifier
- **Improved KNN**: Base paper method with degree-of-attack weighting
- **Proposed KNN**: Novel extensions including **Bounded Distance Compression**, **Adaptive K**, and **Dual Voting**
- **Competitive Baselines**: Random Forest, Support Vector Machines (SVM), XGBoost, LightGBM, Multi-Layer Perceptron (MLP)
- **Multi-Trial Rigor**: 30 independent runs with distinct seeds, Wilcoxon signed-rank tests, Cohen's $d$, and Cliff's $\delta$ effect sizes.

> **Note on SDN**: Mininet/Ryu emulation (`--sdn`) requires Linux root privileges and kernel namespaces, which are disabled in Kaggle containers. This notebook executes the complete offline ML and empirical benchmark suite.


### 1. Environment & Repository Setup
Verify hardware capabilities and confirm the working directory.


In [ ]:
import os
import sys
import psutil
import platform

print(f"OS: {platform.system()} {platform.release()}")
print(f"Python: {sys.version.split()[0]}")
print(f"CPU count: {psutil.cpu_count(logical=True)}")
print(f"RAM: {psutil.virtual_memory().total / (1024**3):.2f} GB")

# Ensure repository root is our working directory
REPO_NAME = "knn-ddos-detection"
REPO_URL = "https://github.com/ihkokil/knn-ddos-detection.git"

if not os.path.exists("main.py"):
    if os.path.exists(os.path.join("..", "main.py")):
        os.chdir("..")
    elif os.path.exists(REPO_NAME):
        os.chdir(REPO_NAME)
    else:
        print(f"Cloning {REPO_NAME} from GitHub...")
        !git clone {REPO_URL}
        if os.path.exists(REPO_NAME):
            os.chdir(REPO_NAME)

print("Working Directory:", os.getcwd())
assert os.path.exists("main.py"), "Error: main.py not found in working directory!"
print("✓ Repository root confirmed.")


### 2. Install Project Dependencies


In [ ]:
# Install required packages
!pip install -q -r requirements.txt
print("✓ Dependencies installed.")


### 3. Quick Pipeline Sanity Check (Synthetic Data)
Runs an end-to-end test on synthetic traffic in a few seconds to verify data preprocessing, feature engineering, and model inference.


In [ ]:
!python main.py --phase full_eval --quick-test


### 4. Comprehensive Empirical Evaluation (30 Trials on NSL-KDD)
Executes the full experimental protocol:
- 30 independent trials with random seeds
- NSL-KDD benchmark with group/time-aware train/test splits
- Proposed KNN vs Improved KNN vs Baseline KNN
- Competitive baselines: Random Forest, SVM, XGBoost, LightGBM, MLP
- Non-parametric statistical hypothesis testing (Wilcoxon, Friedman, Holm-Bonferroni)


In [ ]:
# Run full evaluation (default: NSL-KDD, 30 trials)
# Set --num-trials to 10 for a quicker test or 30 for publication results
!python main.py --phase full_eval --datasets NSL-KDD --num-trials 30 --download-data


### 5. Benchmark Performance Summary Table
Loads the aggregate metrics across all 30 trials.


In [ ]:
import json
import glob
import pandas as pd
from IPython.display import display

results_file = "results/evaluation/evaluation_complete.json"
if os.path.exists(results_file):
    with open(results_file, "r") as f:
        data = json.load(f)
    
    print("Evaluation Run Date:", data.get("timestamp", "N/A"))
    
    for ds_name, ds_results in data.get("datasets", {}).items():
        print(f"\n{'='*30} {ds_name} {'='*30}")
        models = ds_results.get("models", {})
        summary_rows = []
        for model_name, metrics in models.items():
            summary_rows.append({
                "Model": model_name.upper(),
                "Accuracy": f"{metrics.get('accuracy_mean', 0)*100:.2f}% ± {metrics.get('accuracy_std', 0)*100:.2f}%",
                "Precision": f"{metrics.get('precision_mean', 0)*100:.2f}% ± {metrics.get('precision_std', 0)*100:.2f}%",
                "Recall": f"{metrics.get('recall_mean', 0)*100:.2f}% ± {metrics.get('recall_std', 0)*100:.2f}%",
                "F1-Score": f"{metrics.get('f1_mean', 0)*100:.2f}% ± {metrics.get('f1_std', 0)*100:.2f}%",
                "AUC-ROC": f"{metrics.get('auc_roc_mean', 0):.4f} ± {metrics.get('auc_roc_std', 0):.4f}",
                "Latency (ms)": f"{metrics.get('latency_mean_ms', 0):.2f}"
            })
        df = pd.DataFrame(summary_rows)
        display(df)
else:
    csv_files = glob.glob("results/**/metrics/*.csv", recursive=True)
    if csv_files:
        print(f"Found {len(csv_files)} metric CSV files. Showing latest:")
        df = pd.read_csv(csv_files[0])
        display(df.head(10))
    else:
        print("No results file found yet. Make sure step 4 has completed.")


### 6. Statistical Significance Tests & Effect Sizes
Displays non-parametric tests confirming whether Proposed KNN improvements are statistically significant.


In [ ]:
if os.path.exists(results_file):
    with open(results_file, "r") as f:
        data = json.load(f)
    
    for ds_name, ds_results in data.get("datasets", {}).items():
        stats = ds_results.get("statistical_tests", {})
        if stats:
            print(f"\n{'='*25} Statistical Tests for {ds_name} {'='*25}")
            stat_rows = []
            for comp_name, comp_data in stats.items():
                stat_rows.append({
                    "Comparison": comp_name,
                    "Wilcoxon p-val": f"{comp_data.get('wilcoxon_p', 1.0):.4e}",
                    "Significant (p<0.05)": "✓ Yes" if comp_data.get('wilcoxon_p', 1.0) < 0.05 else "✗ No",
                    "Cohen's d": f"{comp_data.get('cohens_d', 0):.3f}",
                    "Effect Magnitude": comp_data.get('effect_magnitude', 'N/A')
                })
            display(pd.DataFrame(stat_rows))
        else:
            print("No statistical tests dictionary found.")


### 7. Generated Visualizations
Displays publication-ready plots generated during the evaluation.


In [ ]:
from IPython.display import Image, display
import glob

plot_files = glob.glob("results/**/plots/*.png", recursive=True) + glob.glob("results/**/*.png", recursive=True)
plot_files = sorted(list(set(plot_files)))

print(f"Found {len(plot_files)} generated figures:")
for p in plot_files:
    print(f"Displaying: {p}")
    display(Image(filename=p))


### 8. Package & Export Results
Compress all generated results, metrics, and figures into a zip file available under the Kaggle Output tab.


In [ ]:
import shutil

output_archive = "knn_ddos_evaluation_results"
shutil.make_archive(output_archive, "zip", "results")
print(f"✓ Successfully created {output_archive}.zip ({os.path.getsize(output_archive + '.zip') / 1024:.1f} KB)")
print("You can download this archive directly from the Kaggle Output panel on the right.")
